In [1]:
import os

import matplotlib.pyplot as plt
import matplotlib
import numpy

from astropy.coordinates import SkyCoord
import astropy.units as u

from rascil.processing_components import plot_uvcoverage, plot_visibility

from radioimaging.util import util
from radioimaging.visibility import vissim, residual, weights, bda, ingest
from radioimaging.images import images
from rascil.processing_components import export_visibility_to_ms

plt.set_loglevel("critical")
cmap='turbo'
matplotlib.rcParams['figure.figsize'] = [5, 5]

In [2]:
gt_name = "../data/SGRA_full_gt.fits"
config_name = "MID"

phasecentre = SkyCoord(ra=+56.0 * u.deg, dec=-15.0 * u.deg, frame='icrs', equinox='J2000')
ha_interval=(-2,2)

bandwidth = 1e4
base_freq = 1e9
nchan = 1
freqs = range(int(base_freq), int(base_freq + bandwidth*nchan), int(bandwidth))
bandwidths = [bandwidth] * len(freqs)

noise_level = 5
num_pix = util.fromfits(gt_name).shape[0]

tmp_ms_names = ["tmp/tmp" + str(i) + ".ms" for i in range(len(freqs))]

decorr_limit = 0.99

output_dataset_name = "../data8/test3.ms"

In [ ]:
for i, freq in enumerate(freqs):
    vcurr = vissim.generate_visibilities(phasecentre,ha_interval,tel=config_name, integration_time=5, frequencies=numpy.array([freq]).astype(float), channel_bandwidths=numpy.array([bandwidth]))
    vcurr, cell_size, im = residual.visibilities_from_image(vcurr, gt_name,return_cellsize=True,return_image=True,scale_factor=1.8)
    print(vcurr.vis.shape)
    vcurr = vissim.add_noise_to_vis(vcurr, noise_level)
    if decorr_limit < 1:
        cell_size_deg = cell_size * 180 / numpy.pi
        field_size_deg = cell_size_deg * num_pix 
        print(cell_size)
        bda.bda_and_export_to_ms2(tmp_ms_names[i], vcurr, decorr_limit, field_size_deg / 2, only_residual_decorr=False)
    else:
        export_visibility_to_ms(output_dataset_name, [vcurr])

In [4]:
datasets_str = ""
for tms in tmp_ms_names:
    datasets_str += tms + " "

join_exec = "python join_datasets.py --idatasets " + datasets_str + "--odataset_name " + output_dataset_name

os.system(join_exec)

2025-12-04 11:04:46	SEVERE	getcell::TIME	Exception Reported: TableProxy::getCell: no such row
2025-12-04 11:04:46	SEVERE	getcell::TIME	Exception Reported: TableProxy::getCell: no such row
2025-12-04 11:04:46	WARN	MSConcat::copySpwAndPol	Negative or zero total bandwidth in SPW 0 of MS to be appended.


['tmp/tmp0.ms', 'tmp/tmp1.ms']


0

In [5]:
for tms in tmp_ms_names:
    os.system("rm -r " + tms)